In [1]:
pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import mysql.connector
import os

# list of csv file and their correspondig table names
csv_files = [
    ('customers.csv', 'customers'),
    ('orders.csv','orders'),
    ('sellers.csv', 'sellers'),
    ('products.csv', 'products'),
    ('geolocation.csv', 'geolocation'),
    ('payments.csv', 'payments') ,
    ('order_items.csv', 'order_items')
    # Added payement.csv for specific handling
]

# connect to the MYSQL dataset
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='password',
    database='ecommers'
)

cursor = conn.cursor()

# Folder containig the csv file 
folder_path = 'D:/E-commerse'

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'INT'
    else:
        return 'TEXT'

for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path,csv_file)

    # read the csv file into pandas dataframe
    df = pd.read_csv(file_path)

    # Replace NaN with None to handale SQL NULL
    df = df.where(pd.notnull(df), None)

    # Debugging: Checl for NaN values 
    print(f"Processing {csv_files}")
    print(f"NaN values befor replacement:\n{df.isnull().sum()}\n")

    # Clean column names
    df.columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_') for col in df.columns]

    # Generate the CREATE TABLE statement with appropriate data types
    columns = ', '.join([f'`{col}` {get_sql_type(df[col].dtype)}' for col in df.columns])
    create_table_query = f'CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})'
    cursor.execute(create_table_query)

    # Insert DataFrame data into the MySQL table
    for _, row in df.iterrows():
        # Convert row to tuple and handle NaN/None explicitly
        values = tuple(None if pd.isna(x) else x for x in row)
        sql = f"INSERT INTO `{table_name}` ({', '.join(['`' + col + '`' for col in df.columns])}) VALUES ({', '.join(['%s'] * len(row))})"
        cursor.execute(sql, values)

    # Commit the transaction for the current CSV file
    conn.commit()

# Close the connection
conn.close()

Processing [('customers.csv', 'customers'), ('orders.csv', 'orders'), ('sellers.csv', 'sellers'), ('products.csv', 'products'), ('geolocation.csv', 'geolocation'), ('payments.csv', 'payments'), ('order_items.csv', 'order_items')]
NaN values befor replacement:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Processing [('customers.csv', 'customers'), ('orders.csv', 'orders'), ('sellers.csv', 'sellers'), ('products.csv', 'products'), ('geolocation.csv', 'geolocation'), ('payments.csv', 'payments'), ('order_items.csv', 'order_items')]
NaN values befor replacement:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64